This notebook is intended to assist with performing pilot studies to determine the correct numer of trials required to get a statistically-meaningful result when running benchmarks. It assumes you have already run (and tagged) samples. It will tehn look at them to determine how large your sample size needs to be to determine if an effect exists.

We begin by setting some variables:

In [ ]:
# α (Alpha) is the boundry for how likely it is that you will have a false positive (find an effect when none exists)
ALPHA=0.05

# Power is roughly the chance that you detect an effect, if one exists.
POWER=0.8

# Tolerable effect is how large an effect/overhead you are willing to consider equivalent, in percent
TOLERABLE_EFFECT=0.005

# The query for your baseline case (we assume everything is being compared to this test case)
baseline_query="test==\"no_monitoring\" and benchmark==\"sp\""

# The queries for your test cases - the key will be used as the name.
experimental_queries={ 
    "HPCPerfStats": "test==\"hpcperfstats\"  and benchmark==\"sp\"",
    "LDMS": "test==\"ldms\"  and benchmark==\"sp\"",
    "BMC_with_helper": "test==\"bmc_with_amsd_clean\" and benchmark==\"sp\"",
    "BMC_no_helper": "test==\"bmc_no_amsd\" and benchmark==\"sp\"",
}

# If needed, this can be used to limit the date range
date_range = "" # for example: now-1d:now

# If needed, any extra arguments for reframe
reframe_extra_args = ""

Next we grab data about the studies and transform it to an easy to work with data structure:

In [ ]:
from json import loads

json={}
tagsets = { "baseline": baseline_query } | experimental_queries

for key in tagsets:
    # Call reframe to get the JSON representation of all of our baseline jobs.
    jres = ! ./reframe.sh {reframe_extra_args} --describe-stored-sessions '{date_range}?{tagsets[key]}'
    json[key] = loads('\n'.join(jres)) # Join it into one string

from pprint import pprint
pprint(json)

import re

perf = {} # Scenario, then name of test, then performance metric, containing a list of test results
metric_name_re = re.compile(r'[^:]+:[^:]+:([^:]+)')

for scenario in json:
    perf[scenario] = {}
    for session in json[scenario]:
        for run in session['runs']:
            for testcase in run['testcases']:
                if testcase['fail_phase'] is None:
                    testname = testcase['name']
                    if testname not in perf[scenario]:
                        perf[scenario][testname] = {}
                    for metric in testcase['perfvalues']:
                        # Do some string processing to get the name of the perf counter
                        metric_name = metric_name_re.match(metric).group(1)
                        # Check if that counter already exists and add if not
                        if metric_name not in perf[scenario][testname]:
                            perf[scenario][testname][metric_name] = {
                                "values": [],
                                "unit": testcase['perfvalues'][metric][4]
                            }
                        perf[scenario][testname][metric_name]['values'].append(
                            testcase['perfvalues'][metric][0]
                        )
pprint(perf)

And finally, for each metric of each experimental scenario, we perform the power test to determine how many tests are necessary. Note that the suggested number of runs is the minium of those required for either statistically demonstrating a difference or showing that the difference (if it exists) is within the equivalence bound. Run this number of times, the tests will most likely have a statistically significant result for one test but not the other. If you have defined your equivalence bound well, this is probably the most useful outcome.

In some cases, samplesize_rank_compare_onetail() fails. When this occurs, it likely indicates the data is already statistically significant.

In [ ]:
from math import sqrt, ceil
from numpy import var, mean, ndarray, array as nparr
from statsmodels.stats.nonparametric import samplesize_rank_compare_onetail
from scipy import stats

scenarios = list(perf.keys())
scenarios.remove('baseline')

for scenario in scenarios:
    print(perf['baseline'].keys())
    for test in perf['baseline'].keys():
        for metric in perf['baseline'][test].keys():
            basedata = perf['baseline'][test][metric]['values']
            exprdata = perf[scenario][test][metric]['values']

            tolerable_effect_offset = mean(basedata) * TOLERABLE_EFFECT

            try:
                res = samplesize_rank_compare_onetail(
                    basedata,exprdata, 
                    alpha=ALPHA, power=POWER
                ).nobs_treat
            except ValueError:
                res=len(basedata)
                print(f"{scenario}: {test}: {metric} difference test may already be statistically significant")
                
            # print(res)
            # Lower half of the OST
            try:
                res_lower = samplesize_rank_compare_onetail(
                        [ x - tolerable_effect_offset for x in basedata ],
                        exprdata, 
                    alpha=ALPHA, power=POWER,
                    alternative='larger'
                ).nobs_treat
            
            except ValueError:
                res_lower=len(basedata)
                print(f"{scenario}: {test}: {metric} one-sided lower test may already be statistically significant")
            # Upper half of the OST

            try:
                res_upper = samplesize_rank_compare_onetail(
                        [ x + tolerable_effect_offset for x in basedata ],
                        exprdata, 
                    alpha=ALPHA, power=POWER,
                    alternative='smaller'
                ).nobs_treat
            except ValueError:
                res_upper=len(basedata)
                print(f"{scenario}: {test}: {metric} one-sided upper difference test may already be statistically significant")

            print(f"""{scenario}: {test}: {metric}: 
    Suggested samples: {ceil(min(res,max(res_lower,res_upper)))}
    Required samples to prove effect: {ceil(res)}
    Required samples for equivalence: {ceil(max(res_upper,res_lower))}""")
                # TTestIndPower().plot_power(
                #     nobs=nparr(range(10,2*ceil(res),10)), alpha=ALPHA,
                #     effect_size=[effect_size]
                # )